# Predicción de Morosidad Temprana

## Objetivos de Aprendizaje

En este notebook, aprenderás a:

1. **Predecir morosidad** en los primeros 90 días con `SNOWFLAKE.ML.CLASSIFICATION`
2. **Identificar señales tempranas** de impago
3. **Calcular probabilidad de default** (PD) por segmento
4. **Crear sistema de alertas** tempranas
5. **Dashboard de monitorización** de cartera

---

## Lo Que Construirás

Un **sistema de alerta temprana de morosidad**:
- Predicción de impago en primeros 90 días post-concesión
- Features de comportamiento temprano (pagos, uso tarjeta, saldos)
- Alertas por nivel de riesgo
- Dashboard de monitorización

**Valor de Negocio**: Intervenir antes del primer impago para reducir provisiones.

---

## Funcionalidades Clave

- `SNOWFLAKE.ML.CLASSIFICATION` — Predicción de morosidad
- Window functions — Tendencias de comportamiento
- `GENERATOR()` — Datos de cartera sintéticos

¡Comencemos!

---

## Paso 1: Configuración del Entorno

In [ ]:
CREATE DATABASE IF NOT EXISTS MOROSIDAD_TEMPRANA_DB;
CREATE SCHEMA IF NOT EXISTS MOROSIDAD_TEMPRANA_DB.PUBLIC;
USE SCHEMA MOROSIDAD_TEMPRANA_DB.PUBLIC;

CREATE WAREHOUSE IF NOT EXISTS COMPUTE_WH 
    WITH WAREHOUSE_SIZE = 'SMALL'
    AUTO_SUSPEND = 60 AUTO_RESUME = TRUE;
USE WAREHOUSE COMPUTE_WH;

---

## Paso 2: Generar Datos de Cartera Reciente

### Qué Vamos a Crear

- **6.000 préstamos** concedidos en últimos 12 meses
- **Comportamiento de pago** de los 3 primeros meses
- **12% de morosidad** (primer impago en primeros 90 días)

In [ ]:
CREATE OR REPLACE TABLE CARTERA_RECIENTE (
    credito_id VARCHAR(10) PRIMARY KEY,
    fecha_concesion DATE,
    importe DECIMAL(12,2),
    tipo_interes DECIMAL(5,3),
    plazo_meses INTEGER,
    segmento VARCHAR(20),
    score_concesion INTEGER,
    dti_concesion FLOAT,
    pago_mes1_puntual BOOLEAN,
    pago_mes2_puntual BOOLEAN,
    pago_mes3_puntual BOOLEAN,
    uso_tarjeta_pct FLOAT,
    saldo_cuenta_medio DECIMAL(10,2),
    es_moroso_90d BOOLEAN
);

INSERT INTO CARTERA_RECIENTE
SELECT
    'CRE' || LPAD(SEQ4()::STRING, 6, '0'),
    DATEADD('day', -UNIFORM(30, 365, RANDOM()), CURRENT_DATE()),
    ROUND(UNIFORM(3000, 100000, RANDOM()), 2),
    ROUND(UNIFORM(3.0, 10.0, RANDOM()), 3),
    ARRAY_CONSTRUCT(12,24,36,48,60)[UNIFORM(0,4,RANDOM())]::INT,
    ARRAY_CONSTRUCT('Premium','Particular','Joven','Empresa')[UNIFORM(0,3,RANDOM())]::VARCHAR,
    UNIFORM(400, 850, RANDOM()),
    ROUND(UNIFORM(0.10, 0.55, RANDOM()), 2),
    CASE WHEN UNIFORM(0,100,RANDOM()) < 88 THEN TRUE ELSE FALSE END,
    CASE WHEN UNIFORM(0,100,RANDOM()) < 85 THEN TRUE ELSE FALSE END,
    CASE WHEN UNIFORM(0,100,RANDOM()) < 82 THEN TRUE ELSE FALSE END,
    ROUND(UNIFORM(0, 95, RANDOM()), 1),
    ROUND(UNIFORM(100, 20000, RANDOM()), 2),
    CASE WHEN UNIFORM(0,100,RANDOM()) < 12 THEN TRUE ELSE FALSE END
FROM TABLE(GENERATOR(ROWCOUNT => 6000));

SELECT es_moroso_90d, COUNT(*), ROUND(AVG(score_concesion),0) AS score_medio
FROM CARTERA_RECIENTE GROUP BY 1;

---

## Paso 3: Ingeniería de Variables y Entrenamiento

### Features Tempranas

- Nº de pagos puntuales en primeros 3 meses
- Score de concesión, DTI
- Uso de tarjeta y saldo medio

In [ ]:
CREATE OR REPLACE TABLE FEATURES_MOROSIDAD AS
SELECT
    credito_id,
    importe::FLOAT,
    tipo_interes::FLOAT,
    plazo_meses::FLOAT,
    score_concesion::FLOAT,
    dti_concesion::FLOAT,
    (pago_mes1_puntual::INT + pago_mes2_puntual::INT + pago_mes3_puntual::INT)::FLOAT AS pagos_puntuales_3m,
    uso_tarjeta_pct::FLOAT,
    saldo_cuenta_medio::FLOAT,
    CASE segmento WHEN 'Premium' THEN 3 WHEN 'Empresa' THEN 2 WHEN 'Particular' THEN 1 ELSE 0 END::FLOAT AS nivel_segmento,
    es_moroso_90d
FROM CARTERA_RECIENTE;

CREATE OR REPLACE TABLE TRAIN_MORO AS SELECT * FROM FEATURES_MOROSIDAD SAMPLE (80);
CREATE OR REPLACE TABLE TEST_MORO AS SELECT * FROM FEATURES_MOROSIDAD WHERE credito_id NOT IN (SELECT credito_id FROM TRAIN_MORO);

CREATE OR REPLACE SNOWFLAKE.ML.CLASSIFICATION predictor_morosidad(
    INPUT_DATA => SYSTEM$REFERENCE('TABLE', 'TRAIN_MORO'),
    TARGET_COLNAME => 'ES_MOROSO_90D'
);

In [ ]:
CALL predictor_morosidad!SHOW_EVALUATION_METRICS();
CALL predictor_morosidad!SHOW_FEATURE_IMPORTANCE();

---

## Paso 4: Alertas Tempranas

In [ ]:
CREATE OR REPLACE TABLE ALERTAS_MOROSIDAD AS
SELECT
    credito_id, importe, score_concesion, pagos_puntuales_3m,
    es_moroso_90d AS moroso_real,
    predictor_morosidad!PREDICT(
        OBJECT_CONSTRUCT(
            'IMPORTE', importe, 'TIPO_INTERES', tipo_interes,
            'PLAZO_MESES', plazo_meses, 'SCORE_CONCESION', score_concesion,
            'DTI_CONCESION', dti_concesion, 'PAGOS_PUNTUALES_3M', pagos_puntuales_3m,
            'USO_TARJETA_PCT', uso_tarjeta_pct,
            'SALDO_CUENTA_MEDIO', saldo_cuenta_medio,
            'NIVEL_SEGMENTO', nivel_segmento
        )
    ) AS pred,
    ROUND(pred['probability']['true']::FLOAT * 100, 1) AS prob_moroso,
    CASE
        WHEN pred['probability']['true']::FLOAT >= 0.6 THEN 'ALTO'
        WHEN pred['probability']['true']::FLOAT >= 0.3 THEN 'MEDIO'
        ELSE 'BAJO'
    END AS riesgo_morosidad
FROM TEST_MORO;

SELECT riesgo_morosidad, COUNT(*) AS n, ROUND(AVG(prob_moroso),1) AS prob_media
FROM ALERTAS_MOROSIDAD GROUP BY 1 ORDER BY prob_media DESC;

---

## Paso 5: Dashboard de Morosidad

In [ ]:
import streamlit as st
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()

st.title('Predicción de Morosidad Temprana')
st.markdown('### Alertas Primeros 90 Días')

total = session.sql('SELECT COUNT(*) FROM ALERTAS_MOROSIDAD').collect()[0][0]
alto = session.sql("SELECT COUNT(*) FROM ALERTAS_MOROSIDAD WHERE riesgo_morosidad='ALTO'").collect()[0][0]
importe_riesgo = session.sql("SELECT ROUND(SUM(importe),0) FROM ALERTAS_MOROSIDAD WHERE riesgo_morosidad='ALTO'").collect()[0][0]

c1, c2, c3 = st.columns(3)
c1.metric('Total Créditos', f'{total:,}')
c2.metric('Riesgo Alto', f'{alto:,}')
c3.metric('Importe en Riesgo', f'{importe_riesgo:,.0f}€')

st.markdown('---')
df = session.sql('SELECT riesgo_morosidad, COUNT(*) AS n FROM ALERTAS_MOROSIDAD GROUP BY 1').to_pandas()
st.bar_chart(df.set_index('RIESGO_MOROSIDAD'))

st.markdown('---')
st.subheader('Créditos de Alto Riesgo')
df_alto = session.sql("SELECT credito_id, importe, prob_moroso, score_concesion, pagos_puntuales_3m FROM ALERTAS_MOROSIDAD WHERE riesgo_morosidad='ALTO' ORDER BY prob_moroso DESC LIMIT 20").to_pandas()
st.dataframe(df_alto)

---

## Paso 6: Limpieza (Opcional)

In [ ]:
-- Descomentar para limpiar

-- DROP DATABASE IF EXISTS MOROSIDAD_TEMPRANA_DB;